In [ ]:
import nltk
import math
import random
from collections import Counter
from nltk.corpus import brown, stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('brown')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
# Initializing the Modules
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
def preprocess_text(text):

    # Normalization (Lowercase)
    text = text.lower()

    # Tokenization
    tokens = word_tokenize(text)

    # Remove punctuation & non-alphabetic tokens
    tokens = [word for word in tokens if word.isalpha()]

    # Stopword Removal
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens

In [ ]:
# Entropy Calculation
def calculate_entropy(word_list):
    total_words = len(word_list)
    freq_dist = Counter(word_list)

    entropy = 0
    for word in freq_dist:
        prob = freq_dist[word] / total_words
        entropy += -prob * math.log2(prob)

    return entropy

In [ ]:
# Cross-Entropy Calculation
def calculate_cross_entropy(p_words, q_words):
    p_total = len(p_words)
    q_total = len(q_words)

    p_freq = Counter(p_words)
    q_freq = Counter(q_words)

    cross_entropy = 0

    for word in p_freq:
        p_prob = p_freq[word] / p_total
        q_prob = q_freq[word] / q_total if word in q_freq else 1e-10

        cross_entropy += -p_prob * math.log2(q_prob)

    return cross_entropy

In [ ]:
# Perplexity Calculation
def calculate_perplexity(entropy):
    return 2 ** entropy

In [ ]:
# For Small Corpus
small_corpus = """
The cats were sitting on the mats.
The dogs are running in the garden.
The cat chased the dogs.
"""
small_tokens = preprocess_text(small_corpus)

random.shuffle(small_tokens)
split_index = int(0.8 * len(small_tokens))

train_small = small_tokens[:split_index]
test_small = small_tokens[split_index:]

entropy_small = calculate_entropy(small_tokens)
cross_entropy_small = calculate_cross_entropy(test_small, train_small)
perplexity_small = calculate_perplexity(entropy_small)

print("Small Corpus (After Preprocessing)")
print("Total Tokens:", len(small_tokens))
print("Entropy:", entropy_small)
print("Cross Entropy:", cross_entropy_small)
print("Perplexity:", perplexity_small)


Small Corpus (After Preprocessing)
Total Tokens: 9
Entropy: 2.725480556997868
Cross Entropy: 18.013317935465615
Perplexity: 6.613805215240195


In [ ]:
# For Brown Corpus
brown_text = " ".join(brown.words())
brown_tokens = preprocess_text(brown_text)

random.shuffle(brown_tokens)
split_index = int(0.8 * len(brown_tokens))

train_brown = brown_tokens[:split_index]
test_brown = brown_tokens[split_index:]

entropy_brown = calculate_entropy(brown_tokens)
cross_entropy_brown = calculate_cross_entropy(test_brown, train_brown)
perplexity_brown = calculate_perplexity(entropy_brown)

print("\nBrown Corpus (After Preprocessing)")
print("Total Tokens:", len(brown_tokens))
print("Entropy:", entropy_brown)
print("Cross Entropy:", cross_entropy_brown)
print("Perplexity:", perplexity_brown)


Brown Corpus (After Preprocessing)
Total Tokens: 515817
Entropy: 12.634001741165058
Cross Entropy: 13.155270892447334
Perplexity: 6356.433229690333


In [ ]:
# Cross-Entropy Calculation
cross_entropy_value = calculate_cross_entropy(small_tokens, brown_tokens)

print("\nCross Entropy (Small || Brown):", cross_entropy_value)
print("Cross Perplexity:", calculate_perplexity(cross_entropy_value))


Cross Entropy (Small || Brown): 13.635153044438683
Cross Perplexity: 12723.015663464505


Experiment 02

In [ ]:
import numpy as np
import re

text_A = "Natural language processing enables computers to understand human language"

text_B = "Language models learn patterns in text and understand language"

def preprocess(text):

    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()

    return tokens


tokens_A = preprocess(text_A)
tokens_B = preprocess(text_B)


vocab = list(set(tokens_A + tokens_B))

def compute_prob(tokens, vocab):

    total = len(tokens)

    probs = {}

    for word in vocab:

        count = tokens.count(word)

        probs[word] = count / total

    return probs


prob_A = compute_prob(tokens_A, vocab)
prob_B = compute_prob(tokens_B, vocab)


def entropy(prob_dist):

    probs = np.array(list(prob_dist.values()))

    probs = probs[probs > 0]

    H = -np.sum(probs * np.log2(probs))

    return H

def cross_entropy(p_dist, q_dist):

    epsilon = 1e-10

    ce = 0

    for word in p_dist:

        p = p_dist[word]

        q = q_dist[word]

        if q == 0:
            q = epsilon

        if p > 0:
            ce += p * np.log2(q)

    return -ce


def perplexity(cross_entropy_value):

    return 2 ** cross_entropy_value


entropy_A = entropy(prob_A)
entropy_B = entropy(prob_B)

cross_AB = cross_entropy(prob_A, prob_B)

perp_AB = perplexity(cross_AB)


print("Tokens A:", tokens_A)
print("Tokens B:", tokens_B)

print("\nEntropy of Corpus A:", entropy_A)
print("Entropy of Corpus B:", entropy_B)

print("\nCross Entropy (A || B):", cross_AB)

print("\nPerplexity:", perp_AB)

Tokens A: ['natural', 'language', 'processing', 'enables', 'computers', 'to', 'understand', 'human', 'language']
Tokens B: ['language', 'models', 'learn', 'patterns', 'in', 'text', 'and', 'understand', 'language']

Entropy of Corpus A: 2.94770277922009
Entropy of Corpus B: 2.94770277922009

Cross Entropy (A || B): 22.9806067441743

Perplexity: 8276599.654616931
